# Kaggle V4: The Agentic Trace Experiment

> **Objective**: Test if a base model natively meta-learns the tool-calling harness purely by fine-tuning on raw agentic traces.
>
> **Fixes in this version:**
> - **Data Transformation Block**: Wraps `toolCall` fields in explicit `<tool_call>` JSON blocks so Gemma's chat template doesn't silently delete them.
> - **Strict Alternation**: Merges consecutive identical roles to satisfy Gemma-2's `user` -> `model` -> `user` constraint.
> - **Meta-Learning Evaluation**: Tests the model on a completely novel tool to prove meta-learning over memorization.

## 1. Setup Environment

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes transformers datasets huggingface_hub trackio inspect-ai inspect-evals
print("✅ Dependencies installed.")

## 2. Configuration

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, login

# Authenticate with Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN not found in Kaggle secrets or environment!')

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

HF_USERNAME = HfApi().whoami()['name']
DATASET_ID = "badlogicgames/pi-mono"
MODEL_ID = "unsloth/gemma-2-2b-it-bnb-4bit"
FINAL_REPO_ID = f"{HF_USERNAME}/fine-tuning-agent-v4-traces"
MAX_SEQ_LENGTH = 2048

print(f"✅ Configured. Final model will be pushed to: {FINAL_REPO_ID}")

## 3. Data Transformation Block

In [ ]:
import json, random
from datasets import Dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

print("Downloading pi-mono traces...")
raw_dir = Path("./pi_mono_raw")
snapshot_download(repo_id=DATASET_ID, repo_type="dataset", allow_patterns=["*.jsonl"], local_dir=str(raw_dir))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def extract_text(parts):
    if isinstance(parts, str): return parts
    if not isinstance(parts, list): return ""
    return "\n".join(str(p.get("text", "")) for p in parts if isinstance(p, dict) and p.get("type") == "text").strip()

def convert_event_to_message(event):
    if event.get("type") != "message": return None
    raw = event.get("message") or {}
    role = raw.get("role")
    
    if role == "user":
        content = extract_text(raw.get("content"))
        return {"role": "user", "content": content} if content else None
        
    if role == "assistant":
        parts = raw.get("content") or []
        text = extract_text(parts)
        tc_strs = []
        
        # SERIALIZE TOOL CALLS SO GEMMA DOESN'T DELETE THEM
        if isinstance(parts, list):
            for p in parts:
                if isinstance(p, dict) and p.get("type") == "toolCall":
                    tc = {"name": p.get("name"), "arguments": p.get("arguments") or {}}
                    tc_strs.append(json.dumps(tc))
        
        if tc_strs:
            text += "\n<tool_call>\n" + "\n".join(tc_strs) + "\n</tool_call>"
            
        return {"role": "assistant", "content": text.strip()} if text.strip() else None
        
    if role == "toolResult":
        # MAP TO 'USER' TO SATISFY GEMMA ALTERNATION RULES
        name = raw.get("toolName", "unknown")
        content = extract_text(raw.get("content")) or "[empty output]"
        return {"role": "user", "content": f"<tool_result name=\"{name}\">\n{content}\n</tool_result>"}
        
    return None

examples = []
for path in list(raw_dir.glob("*.jsonl"))[:100]: # Limit for speed, remove [:100] for full dataset
    events = [json.loads(line) for line in path.read_text(errors="replace").splitlines() if line.strip()]
    conv = []
    
    for e in events:
        msg = convert_event_to_message(e)
        if not msg: continue
        # MERGE CONSECUTIVE ROLES TO PREVENT TEMPLATE CRASHES
        if conv and conv[-1]["role"] == msg["role"]:
            conv[-1]["content"] += "\n\n" + msg["content"]
        else:
            conv.append(msg)
            
    if not conv or conv[0]["role"] != "user":
        conv.insert(0, {"role": "user", "content": "Continue the coding-agent session."})
            
    for i in range(1, len(conv)):
        if conv[i]["role"] == "assistant":
            try:
                prompt = tokenizer.apply_chat_template(conv[:i], tokenize=False, add_generation_prompt=True)
                full = tokenizer.apply_chat_template(conv[:i+1], tokenize=False, add_generation_prompt=False)
                if full.startswith(prompt):
                    comp = full[len(prompt):]
                    if len(comp.strip()) > 5 and len(tokenizer(prompt+comp)["input_ids"]) <= MAX_SEQ_LENGTH:
                        examples.append({"prompt": prompt, "completion": comp})
            except Exception: 
                pass

random.Random(42).shuffle(examples)
eval_size = max(1, len(examples) // 20)
train_ds = Dataset.from_list(examples[eval_size:])
eval_ds = Dataset.from_list(examples[:eval_size])
print(f"✅ Transformed {len(examples)} examples without data loss (Train: {len(train_ds)}, Eval: {len(eval_ds)})")

## 4. Training (Full Epoch)

In [ ]:
import gc, torch
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

sft_config = SFTConfig(
    output_dir="./results/v4_traces",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_text_field=None,
    learning_rate=1e-4,
    num_train_epochs=1.0, 
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds, processing_class=tokenizer)
trainer.train()
trainer.save_model(sft_config.output_dir)
del model, trainer
gc.collect(); torch.cuda.empty_cache()

## 5. Merging to Hub

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch
print("Merging weights to fp16...")
base_model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base_model, "./results/v4_traces").merge_and_unload()
merged.save_pretrained("./final_model", safe_serialization=True)
tokenizer.save_pretrained("./final_model")

print(f"Pushing to {FINAL_REPO_ID}...")
HfApi().create_repo(FINAL_REPO_ID, token=HF_TOKEN, exist_ok=True)
HfApi().upload_folder(folder_path="./final_model", repo_id=FINAL_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(FINAL_REPO_ID, token=HF_TOKEN)
print("✅ Push complete!")

## 6. The Meta-Learning Evaluation Suite
Here we test if the model actually learned the *concept* of tool calling rather than just memorizing `bash` or `read`.

In [ ]:
import textwrap
print("=== META-LEARNING EVALUATION SUITE ===\n")

test_cases = [
    {
        "name": "1. Known Tool Recognition",
        "prompt": "List all the python files in the current directory."
    },
    {
        "name": "2. Zero-Shot Meta-Learning (Novel Tool)",
        "prompt": "I need you to use the 'calculate_warp_drive_metrics' tool. The ship has a speed of 'warp-5' and a mass of 1500."
    },
    {
        "name": "3. Implicit Tool Discovery & Schema Adherence",
        "prompt": "We have a breach! Isolate the network using the 'firewall_lockdown' tool on port 8080 immediately."
    }
]

for tc in test_cases:
    print(f"\n--- Test: {tc['name']} ---")
    print(f"PROMPT: {tc['prompt']}")
    
    # Standard format mirroring the pi-mono training traces
    messages = [{"role": "user", "content": tc['prompt']}]
    p = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    out = merged.generate(**tokenizer(p, return_tensors="pt").to(merged.device), max_new_tokens=150)
    response = tokenizer.decode(out[0][len(tokenizer(p)["input_ids"]):], skip_special_tokens=True)
    
    print("\nMODEL OUTPUT:")
    print(textwrap.indent(response, "    "))
    
    if "<tool_call>" in response:
        print("\n✅ SUCCESS: Model correctly generated a tool call block.")
    else:
        print("\n❌ FAILURE: Model failed to use the agentic harness.")
    print("="*50)

## 7. Standard Evaluation (BFCL)

In [ ]:
print("Running Standard BFCL Benchmark...")
print("Note: Since our model uses explicit `<tool_call>` tags rather than native template tools, ")
print("this validates how the framework parses raw text output into tool calls.")
os.environ['FINAL_REPO_ID'] = FINAL_REPO_ID
!inspect eval inspect_evals/bfcl --model hf/$FINAL_REPO_ID --limit 50 --sandbox local